# Exercise 5 - Predictive Maintenance

Practise RUL regression, threshold extrapolation, alarm design and cost decisions on newly simulated data.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/ex-05-predictive-maintenance.ipynb)

- Lecture: [Chapter 5 - Predictive Maintenance with Linear Models](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-05-predictive-maintainance.md)
- Labs: [LM501](https://rathachai.github.io/DA-LAB/learnings/linear/lm501.html) · [LM502](https://rathachai.github.io/DA-LAB/learnings/linear/lm502.html) · [LM503](https://rathachai.github.io/DA-LAB/learnings/linear/lm503.html) · [LM504](https://rathachai.github.io/DA-LAB/learnings/linear/lm504.html)

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## How to use
1. Run the SETUP cells first. They simulate new data (seeded, so everyone gets the same numbers).
2. Each exercise has a blue box with the task, followed by empty code cells. Write your code there.
3. Use **pandas** for tables, **numpy** for calculations, **scikit-learn** for models and metrics, **matplotlib** for plots.
4. Try the hint only after you have tried yourself. Compare your results with the lecture and the demo notebook (nb-05).

## SETUP
The next cell creates four datasets: `fleet` (sensors and RUL for 30 machines), `deg` (a health index over time), `sig` (a vibration-like signal) and `cost_df` (actual and predicted RUL for 150 machines).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
rng = np.random.default_rng(2025)

# (i) fleet: 30 machines x 20 snapshots, 3 sensors + RUL (h)
rows = []
for m in range(1, 31):
    life = rng.uniform(250, 450)                        # total life of this machine (h)
    age = np.sort(rng.uniform(0, life * 0.9, 20))       # 20 snapshots
    rul = life - age
    off1, off2 = rng.normal(0, 2), rng.normal(0, 3)     # machine "fingerprint"
    a1 = 2 + 4 * (age / life) + off1 * 0.3 + rng.normal(0, 0.3, 20)
    a2 = 40 + 30 * (age / life) ** 2 + off2 + rng.normal(0, 1.5, 20)
    a3 = 1500 + rng.normal(0, 40, 20) + off1 * 10
    rows.append(pd.DataFrame({"machine": m, "a1": a1, "a2": a2, "a3": a3, "rul": rul}))
fleet = pd.concat(rows, ignore_index=True).round(2)

# (ii) deg: health index of one bearing, noisy and slightly curved
t = np.arange(0, 161)
true_failure_time = 235
deg = pd.DataFrame({"t": t, "health": (100 * (1 - (t / true_failure_time) ** 1.15) + rng.normal(0, 1.5, t.size)).round(2)})

# (iii) sig: healthy periodic signal, drift fault from a known time
ts = np.arange(300)
fault_start = 220
x = 50 + 5 * np.sin(2 * np.pi * ts / 25) + rng.normal(0, 0.4, ts.size)
x = x + np.where(ts >= fault_start, 0.12 * (ts - fault_start), 0)
sig = pd.DataFrame({"t": ts, "signal": x.round(2)})

# (iv) cost_df: 150 machines, actual vs predicted RUL (h)
actual = rng.uniform(20, 400, 150)
cost_df = pd.DataFrame({"machine_id": np.arange(1, 151), "actual_rul": actual.round(1),
                        "predicted_rul": np.clip(actual * 0.95 + rng.normal(0, 30, 150), 0, None).round(1)})

for name, d_ in [("fleet", fleet), ("deg", deg), ("sig", sig), ("cost_df", cost_df)]:
    print(name, d_.shape); display(d_.head(3))

### Exercise 1 - RUL regression: split by machine vs by row

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 1.** Use `fleet`. Fit a `LinearRegression` that predicts `rul` from `a1, a2, a3` (scale them with `StandardScaler`).<br>
**Compute:** the test MAE with (a) a random split by row and (b) a split by machine (25 % test). Repeat for 10 seeds and report the mean of each. Put the results in a small DataFrame and say which one is more honest.<br>
*Hint:* `train_test_split`, `GroupShuffleSplit(test_size=0.25, random_state=seed).split(X, y, groups)`, `mean_absolute_error`.

</div>

### Exercise 2 - Hand calculation: threshold crossing

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 2.** A bearing's health index is **80** at **t = 100 h** and falls by **0.4 points/h**. The repair threshold is **theta = 50**.<br>
**Compute by hand, then check in code:** (a) the line h(t) = b0 + b1 t, (b) the time when the index reaches theta, (c) the predicted failure time (index = 0), (d) the RUL now.<br>
*Hint:* solve `theta = b0 + b1 t` for `t`, i.e. `t = (theta - b0) / b1`.

</div>

### Exercise 3 - Trend extrapolation on real-looking data

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 3.** Use `deg`. Pretend 'now' is **t = 150 h**. Fit a straight line to the last **L = 30** readings with `LinearRegression`, extrapolate to health = 0, and report the predicted failure time and RUL. Compare with `true_failure_time` (RUL error in hours). Plot the data, the fitted line and the true failure time. Is the line optimistic or pessimistic? Try L = 15 and L = 60.<br>
*Hint:* `deg[deg.t > 150 - L]`, `lr.intercept_`, `lr.coef_[0]`, `plt.axvline`.

</div>

### Exercise 4 - Alarm design on a signal

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 4.** Use `sig` (the drift starts at `fault_start`). Fit an **AR(w = 5)** model on the first **180** points with `LinearRegression` on lagged columns, compute residuals and `sigma` from the training residuals.<br>
**Compute:** for **k = 2, 3, 4** (alarm when |residual| > k * sigma for 3 samples in a row) the **detection delay** and the number of **false alarms** (alarms after t = 180 but before `fault_start`). Put the results in a DataFrame and plot the residuals with the alarms for one k.<br>
*Hint:* `pd.Series.shift(i)` builds lag columns; `.rolling(3).sum() == 3` finds 3 in a row.

</div>

### Exercise 5 - Total cost and the best threshold

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 5.** Use `cost_df` with **Cp = 1000** (repair), **Cw = 5** per hour of wasted life, **Cf = 8000** (breakdown) and inspection interval **D = 50 h**. Rule: repair now if `predicted_rul <= H`; otherwise run on, and the machine breaks down if `actual_rul < D`.<br>
**Compute:** (a) the total cost for **H = 100**; (b) the total cost for H = 0, 10, ..., 300 and the best H; (c) a plot of cost vs H.<br>
*Hint:* `np.where(cond1, cost1, np.where(cond2, cost2, 0))`, `.sum()`, `pd.Series.idxmin()`.

</div>

### Exercise 6 - Why can a pessimistic model be cheaper?

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 6.** Explain in your own words why a *pessimistic* model (predicts too little life) can have a **higher RMSE** but a **lower total cost** than an unbiased model. Use the numbers of Exercise 5 (Cp, Cw, Cf) in your answer. You can test it: shift `predicted_rul` by -20 h and recompute RMSE and cost.<br>
*Hint:* think about which mistake is more expensive, and the safety factor in design.

</div>

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Your answer:**

(write here)

</div>

### Exercise 7 - Optional challenge: expected cost with a normal failure-time model

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 7.** A machine's failure time T is normal with mean **mu = 300 h** and standard deviation **SE = 40 h**. Costs: **Cp = 1000, Cw = 20 per h, Cf = 10000**. If repair is planned at time `tp`, the expected cost is
`P(T <= tp)*Cf + P(T > tp)*Cp + Cw * E[(T - tp)+]`.<br>
**Compute:** the expected cost for `tp` from 150 to 300 h (step 5), plot it, and find the best `tp`. What breakdown probability does the best plan accept?<br>
*Hint:* simulate with `rng.normal(mu, SE, 100000)` and average, or use `math.erf` for the normal CDF.

</div>

## Done
Compare your answers with the demo notebook and discuss them in class.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io